# Pré-processamento de dados

Esse notebook realiza o pré-processamento dos dados brutos obtidos nos experimentos e transforma essas informações em arquivos `csv` consumidos pelos notebooks de análise.

In [1]:
import os
import json
import re
from pathlib import Path

import pandas as pd

BASE_DIR   = Path("../data")
OUTPUT_DIR = Path("../data_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Funções utilitárias

Conjunto de helpers para ler arquivos `json`, extrair e formatar nomes das máquinas, estratégia de particionamento, entre outros.

In [2]:
def safe_get(d, *keys):
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d


def load_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def load_jsonl_normalized(path):
    try:
        df = pd.read_json(path, lines=True)

        if df.empty:
            return pd.DataFrame()

        dfs = []

        if "metadata" in df.columns:
            dfs.append(pd.json_normalize(df["metadata"]))

        if "metrics" in df.columns:
            dfs.append(pd.json_normalize(df["metrics"]))

        if not dfs:
            return pd.DataFrame()

        return pd.concat(dfs, axis=1)

    except Exception as e:
        print(f"Error reading JSONL: {path}\n{e}")
        return pd.DataFrame()


def extract_machine_name(log_text):
    if not log_text:
        return "Unknown"

    match = re.search(r"Node\(s\)\s*:\s*([^\n]+)", log_text)
    if match:
        return match.group(1).strip()

    match = re.search(r"SubmitHost\s*:\s*([^\n]+)", log_text)
    if match:
        return match.group(1).strip()

    return "Unknown"


def detect_parallelism(name):
    parts = name.upper().split("_")

    if "TP" in parts:
        return "TP"

    if "PP" in parts:
        return "PP"

    return "Single GPU"


def format_experiment_name(name):
    if not isinstance(name, str):
        return str(name)

    parts = name.split("_")

    try:
        node_part = parts[0]
        machine   = parts[1].upper()
        strategy  = parts[2]
        workload  = parts[3]

        n_nodes = int(node_part.replace("N", ""))

        strategy_map = {
            "none": "Single GPU",
            "TP":   "Tensor Parallel",
            "PP":   "Pipeline Parallel",
        }

        strategy_name = strategy_map.get(strategy, strategy)
        workload_name = workload.capitalize()

        return (
            f"{machine} ({n_nodes} node{'s' if n_nodes > 1 else ''})\n"
            f"{strategy_name}\n"
            f"{workload_name}"
        )

    except Exception:
        return name


def find_experiment_files(experiment_path):
    experiment_path = Path(experiment_path)

    files = {
        "profile":        None,
        "jsonl":          None,
        "telemetry":      [],
        "server_metrics": None,
        "inputs":         None,
        "err_logs":       [],
        "out_logs":       [],
    }

    for root, _, filenames in os.walk(experiment_path):
        root = Path(root)

        for file in filenames:
            full_path = root / file

            if file == "profile_export_aiperf.json":
                files["profile"] = full_path

            elif file == "profile_export.jsonl":
                files["jsonl"] = full_path

            elif file == "server_metrics_export.json":
                files["server_metrics"] = full_path

            elif file == "inputs.json":
                files["inputs"] = full_path

            elif file == "telemetry.csv":
                files["telemetry"].append(full_path)

            elif file.endswith(".err"):
                files["err_logs"].append(full_path)

            elif file.endswith(".out"):
                files["out_logs"].append(full_path)

    return files

## Carregador de experimentos

A função `load_experiment` agrupa todos os dados de um único experimento em um dicionário:

| chave            | conteúdo                                                        |
| ---------------- | --------------------------------------------------------------- |
| `profile`        | métricas agregadas exportadas pelo `aiperf`                     |
| `jsonl`          | métricas por requisição                                         |
| `telemetry`      | dicionário com a telemetria por nó                              |
| `server_metrics` | métricas do servidor de inferência                              |
| `inputs`         | prompts/inputs usados na carga                                  |
| `err_logs` / `out_logs` | logs do Slurm                                            |
| `parallelism`    | estratégia detectada a partir do nome (`Single GPU`/`TP`/`PP`)  |
| `machine_name`   | nome do nó extraído dos logs                                    |

In [3]:
def load_experiment(experiment_path):
    experiment_path = Path(experiment_path)
    experiment_name = experiment_path.name

    print(f"Processing: {experiment_name}")

    files = find_experiment_files(experiment_path)

    if files["profile"] is None:
        print(f"  Skipping: no profile_export_aiperf.json")
        return None

    profile        = load_json(files["profile"])
    jsonl_df       = load_jsonl_normalized(files["jsonl"]) if files["jsonl"] else pd.DataFrame()
    server_metrics = load_json(files["server_metrics"])    if files["server_metrics"] else None
    inputs         = load_json(files["inputs"])            if files["inputs"] else None

    # --- Telemetry ---
    telemetry_data = {}
    for telemetry_file in files["telemetry"]:
        try:
            tdf = pd.read_csv(telemetry_file)
            tdf.columns = [c.strip() for c in tdf.columns]
            telemetry_data[telemetry_file.parent.name] = tdf
        except Exception as e:
            print(f"  Error loading telemetry: {telemetry_file}\n  {e}")

    # --- Logs ---
    err_logs, out_logs = {}, {}

    for f in files["err_logs"]:
        try:
            err_logs[f.name] = f.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            pass

    for f in files["out_logs"]:
        try:
            out_logs[f.name] = f.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            pass

    # --- Machine name ---
    machine_name = "Unknown"
    if out_logs:
        machine_name = extract_machine_name(next(iter(out_logs.values())))

    # --- Parallelism (single source of truth) ---
    parallelism = detect_parallelism(experiment_name)

    print(f"  parallelism={parallelism!r}  machine={machine_name!r}")

    return {
        "experiment_name": experiment_name,
        "pretty_name":     format_experiment_name(experiment_name),
        "parallelism":     parallelism,
        "machine_name":    machine_name,

        "profile":        profile,
        "jsonl":          jsonl_df,
        "telemetry":      telemetry_data,
        "server_metrics": server_metrics,
        "inputs":         inputs,

        "err_logs": err_logs,
        "out_logs": out_logs,
    }

Coleta os dados de todos os experimentos.

In [4]:
experiment_paths = sorted([
    p for p in BASE_DIR.iterdir()
    if p.is_dir() and p.name.startswith("N")
])

loaded_experiments = []

for exp_path in experiment_paths:
    exp = load_experiment(exp_path)

    if exp is None:
        continue

    loaded_experiments.append(exp)

print(f"\nLoaded {len(loaded_experiments)} experiments into memory.")

Processing: N1_tupi_none_long_r1_780636
  parallelism='Single GPU'  machine='tupi3'
Processing: N1_tupi_none_short_r1_780637
  parallelism='Single GPU'  machine='tupi3'
Processing: N2_poti_PP_long_r1_780630
  parallelism='PP'  machine='poti[3-4]'
Processing: N2_poti_PP_short_r1_780628
  parallelism='PP'  machine='poti[3-4]'
Processing: N2_poti_TP_long_r1_780629
  parallelism='TP'  machine='poti[1-2]'
Processing: N2_poti_TP_short_r1_780627
  parallelism='TP'  machine='poti[1-2]'
Processing: N2_tupi_PP_long_r1_781237
  parallelism='PP'  machine='tupi[3,5]'
Processing: N2_tupi_PP_short_r1_781238
  parallelism='PP'  machine='tupi[3,5]'
Processing: N2_tupi_TP_long_r1_781236
  parallelism='TP'  machine='tupi[3,5]'
Processing: N2_tupi_TP_short_r1_781235
  parallelism='TP'  machine='tupi[3,5]'
Processing: N4_poti_PP_long_r1_780638
  parallelism='PP'  machine='poti[1-4]'
Processing: N4_poti_PP_short_r1_780634
  parallelism='PP'  machine='poti[1-4]'
Processing: N4_poti_TP_long_r1_780632
  parall

## Métricas de desempenho

Extrai as métricas de desempenho do servidor de inferência a partir do `profile_export_aiperf.json` e do `profile_export.jsonl`.

In [5]:
def extract_metrics(exp):
    profile  = exp["profile"]
    df_jsonl = exp["jsonl"]

    if not profile:
        return None

    def m(name, field):
        return safe_get(profile, name, field)

    parallelism      = exp["parallelism"]
    is_communication = parallelism != "Single GPU"

    metrics = {
        # --- Experiment info ---
        "experiment_name":  exp["experiment_name"],
        "display_name":     exp["pretty_name"],
        "parallelism":      parallelism,
        "machine_name":     exp["machine_name"],
        "is_communication": is_communication,

        # --- Main metrics ---
        "request_throughput_avg":       m("request_throughput",               "avg"),
        "request_latency_avg_ms":       m("request_latency",                  "avg"),
        "request_latency_std_ms":       m("request_latency",                  "std"),
        "time_to_first_token_avg_ms":   m("time_to_first_token",              "avg"),
        "time_to_first_token_std_ms":   m("time_to_first_token",              "std"),
        "inter_token_latency_avg_ms":   m("inter_token_latency",              "avg"),
        "inter_token_latency_std_ms":   m("inter_token_latency",              "std"),
        "time_to_second_token_avg_ms":  m("time_to_second_token",             "avg"),
        "time_to_second_token_std_ms":  m("time_to_second_token",             "std"),
        "output_token_throughput_avg":  m("output_token_throughput_per_user", "avg"),
        "output_token_throughput_std":  m("output_token_throughput_per_user", "std"),
    }

    # --- Percentiles ---
    if not df_jsonl.empty:

        def add_percentiles(col_name, prefix):
            if col_name not in df_jsonl.columns:
                return

            series = pd.to_numeric(
                df_jsonl[col_name],
                errors="coerce"
            ).dropna()

            if series.empty:
                return

            metrics[f"{prefix}_p50"] = series.quantile(0.50)
            metrics[f"{prefix}_p90"] = series.quantile(0.90)
            metrics[f"{prefix}_p99"] = series.quantile(0.99)

        add_percentiles("request_latency.value",      "request_latency_ms")
        add_percentiles("time_to_first_token.value",  "ttft_ms")
        add_percentiles("time_to_second_token.value", "ttst_ms")
        add_percentiles("inter_token_latency.value",  "inter_token_latency_ms")

    # --- Token lengths ---
    if "input_sequence_length.value" in df_jsonl.columns:
        metrics["input_sequence_length_avg"] = df_jsonl["input_sequence_length.value"].mean()
        metrics["input_sequence_length_std"] = df_jsonl["input_sequence_length.value"].std()

    if "output_sequence_length.value" in df_jsonl.columns:
        metrics["output_sequence_length_avg"] = df_jsonl["output_sequence_length.value"].mean()
        metrics["output_sequence_length_std"] = df_jsonl["output_sequence_length.value"].std()

    # --- Little's Law ---
    throughput = metrics["request_throughput_avg"]
    latency    = metrics["request_latency_avg_ms"]

    metrics["avg_concurrency"] = (
        throughput * latency / 1000.0
        if throughput and latency else None
    )

    return metrics

Itera sobre os experimentos já carregados e monta a lista de métricas de desempenho.

In [6]:
results = []

for exp in loaded_experiments:
    metrics = extract_metrics(exp)

    if metrics:
        results.append(metrics)

print(f"Extracted performance metrics for {len(results)} experiments.")

Extracted performance metrics for 14 experiments.


Monta o DataFrame, ordena por estratégia de particionamento e salva como `summary.csv`.

In [7]:
df = pd.DataFrame(results)

if not df.empty:
    ordering = {"Single GPU": 0, "TP": 1, "PP": 2}

    df["_ord"] = df["parallelism"].map(ordering).fillna(999)

    df = (
        df
        .sort_values(["_ord", "experiment_name"])
        .drop(columns=["_ord"])
    )

    df["is_communication"] = df["is_communication"].astype(bool)

    summary_csv = OUTPUT_DIR / "summary.csv"
    df.to_csv(summary_csv, index=False)

    print(f"Saved: {summary_csv.resolve()}")
    print(f"Shape: {df.shape}")

    df[["experiment_name", "parallelism", "is_communication"]]
else:
    print("No valid experiments found.")

Saved: /home/matheus/projects/comp-sys-perf-analysis/data/processed/summary.csv
Shape: (14, 33)


## Métricas das GPUs

Resume a telemetria de cada nó em estatísticas por experimento e nó.

In [8]:
def extract_gpu_metrics(exp):
    telemetry = exp["telemetry"]

    rows = []

    for node_name, df in telemetry.items():

        if df.empty:
            continue

        row = {
            "experiment_name": exp["experiment_name"],
            "display_name":    exp["pretty_name"],
            "parallelism":     exp["parallelism"],
            "machine_name":    exp["machine_name"],
            "node_name":       node_name,
        }

        # GPU UTILIZATION
        if "utilization.gpu [%]" in df.columns:
            series = pd.to_numeric(df["utilization.gpu [%]"], errors="coerce").dropna()

            if not series.empty:
                row["gpu_util_avg"] = series.mean()
                row["gpu_util_std"] = series.std()
                row["gpu_util_p50"] = series.quantile(0.50)
                row["gpu_util_p90"] = series.quantile(0.90)
                row["gpu_util_p99"] = series.quantile(0.99)

        # MEMORY
        if "memory.used [MiB]" in df.columns:
            series = pd.to_numeric(df["memory.used [MiB]"], errors="coerce").dropna()

            if not series.empty:
                row["memory_used_avg_mib"] = series.mean()
                row["memory_used_std_mib"] = series.std()
                row["memory_used_p90_mib"] = series.quantile(0.90)

        # POWER
        if "power.draw [W]" in df.columns:
            series = pd.to_numeric(df["power.draw [W]"], errors="coerce").dropna()

            if not series.empty:
                row["power_draw_avg_w"] = series.mean()
                row["power_draw_std_w"] = series.std()
                row["power_draw_p90_w"] = series.quantile(0.90)

        # TEMPERATURE
        if "temperature.gpu" in df.columns:
            series = pd.to_numeric(df["temperature.gpu"], errors="coerce").dropna()

            if not series.empty:
                row["temperature_avg_c"] = series.mean()
                row["temperature_std_c"] = series.std()
                row["temperature_p90_c"] = series.quantile(0.90)

        rows.append(row)

    return rows

Extrai métricas agregadas de GPU para cada par `(experimento, nó)`.


In [9]:
gpu_results = []

for exp in loaded_experiments:
    gpu_metrics = extract_gpu_metrics(exp)

    if gpu_metrics:
        gpu_results.extend(gpu_metrics)

print(f"Extracted GPU metrics for {len(gpu_results)} (experiment, node) pairs.")

Extracted GPU metrics for 34 (experiment, node) pairs.


Monta o DataFrame e salva como `gpu_summary.csv`.


In [10]:
gpu_df = pd.DataFrame(gpu_results)

if not gpu_df.empty:
    gpu_csv = OUTPUT_DIR / "gpu_summary.csv"
    gpu_df.to_csv(gpu_csv, index=False)

    print(f"Saved: {gpu_csv.resolve()}")
    print(f"Shape: {gpu_df.shape}")
else:
    print("No GPU metrics extracted.")

Saved: /home/matheus/projects/comp-sys-perf-analysis/data/processed/gpu_summary.csv
Shape: (34, 19)


## Série temporal das GPUs

Mantém a granularidade temporal completa da telemetria: cada linha do `telemetry.csv` original vira uma linha no DataFrame final, anotada com os metadados do experimento e do nó.

In [11]:
def extract_gpu_timeseries(exp):
    telemetry = exp["telemetry"]

    rows = []

    for node_name, df in telemetry.items():

        if df.empty:
            continue

        df = df.copy()
        df.columns = [c.strip() for c in df.columns]

        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

        for _, row_df in df.iterrows():
            row = {
                "experiment_name": exp["experiment_name"],
                "display_name":    exp["pretty_name"],
                "parallelism":     exp["parallelism"],
                "machine_name":    exp["machine_name"],
                "node_name":       node_name,
            }

            if "timestamp" in df.columns:
                row["timestamp"] = row_df.get("timestamp")

            if "utilization.gpu [%]" in df.columns:
                row["gpu_utilization"] = pd.to_numeric(
                    row_df.get("utilization.gpu [%]"), errors="coerce"
                )

            if "memory.used [MiB]" in df.columns:
                row["memory_used_mib"] = pd.to_numeric(
                    row_df.get("memory.used [MiB]"), errors="coerce"
                )

            if "power.draw [W]" in df.columns:
                row["power_draw_w"] = pd.to_numeric(
                    row_df.get("power.draw [W]"), errors="coerce"
                )

            if "temperature.gpu" in df.columns:
                row["temperature_gpu_c"] = pd.to_numeric(
                    row_df.get("temperature.gpu"), errors="coerce"
                )

            rows.append(row)

    return rows

Extrai a série temporal completa de cada experimento.


In [12]:
gpu_timeseries = []

for exp in loaded_experiments:
    gpu_ts = extract_gpu_timeseries(exp)

    if gpu_ts:
        gpu_timeseries.extend(gpu_ts)

print(f"Extracted {len(gpu_timeseries)} GPU timeseries samples.")

Extracted 218418 GPU timeseries samples.


Ordena por experimento, nó e timestamp e salva como `gpu_timeseries.csv`.

In [13]:
gpu_ts_df = pd.DataFrame(gpu_timeseries)

if not gpu_ts_df.empty:
    gpu_ts_df = gpu_ts_df.sort_values(["experiment_name", "node_name", "timestamp"])

    gpu_ts_csv = OUTPUT_DIR / "gpu_timeseries.csv"
    gpu_ts_df.to_csv(gpu_ts_csv, index=False)

    print(f"Saved: {gpu_ts_csv.resolve()}")
    print(f"Shape: {gpu_ts_df.shape}")
else:
    print("No GPU timeseries extracted.")

Saved: /home/matheus/projects/comp-sys-perf-analysis/data/processed/gpu_timeseries.csv
Shape: (218418, 10)
